# MIKAGE V2 Real Colab Worker

One-cell worker for the shared Drive queue contract.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import base64
import io
import json
import os
import subprocess
import sys
import time
import traceback
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

def iso_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def ensure_package(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

ensure_package('torch')
ensure_package('diffusers')
ensure_package('transformers')
ensure_package('accelerate')
ensure_package('safetensors')
ensure_package('Pillow', 'PIL')

import torch
from diffusers import AutoPipelineForText2Image

ROOT = Path('/content/drive/MyDrive/mikage_runner')
JOB_INBOX = ROOT / 'job_inbox'
CLAIMS = ROOT / 'claims'
OUTPUTS = ROOT / 'outputs'

for folder in [ROOT, JOB_INBOX, CLAIMS, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print('[MIKAGE] Shared root:', ROOT)
print('[MIKAGE] job_inbox exists:', JOB_INBOX.exists())
print('[MIKAGE] claims exists:', CLAIMS.exists())
print('[MIKAGE] outputs exists:', OUTPUTS.exists())

MODEL_ID = os.environ.get('MIKAGE_COLAB_MODEL_ID', 'stabilityai/sdxl-turbo')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print('[MIKAGE] device =', DEVICE)
if DEVICE == 'cuda':
    print('[MIKAGE] gpu =', torch.cuda.get_device_name(0))

pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=DTYPE, variant='fp16' if DEVICE == 'cuda' else None)
pipe = pipe.to(DEVICE)
if hasattr(pipe, 'safety_checker'):
    pipe.safety_checker = None
print('[MIKAGE] model loaded:', MODEL_ID)

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY') or ''
GEMINI_MODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash')
JUDGE_PROMPT = (
    'You are MIKAGE visual quality judge. Evaluate one rendered image for canon-aware image quality. '
    'Return strict JSON only with keys: source, status, quality_score, overall_score, failure_codes, notes. '
    'Allowed status: PASS, REVIEW, REJECT. Allowed failure_codes: OBJECT_UNREADABLE, ABSTRACT_COMPOSITION, '
    'TEXTURE_ONLY_FRAME, CERAMIC_NOT_CONVINCING, CANON_DRIFT, SILHOUETTE_BREAK, COMPOSITION_COLLAPSE. '
    'Use source=live. quality_score and overall_score must be 0..1.'
)

def safe_read_json(file_path):
    with open(file_path, 'r', encoding='utf-8-sig') as handle:
        return json.load(handle)

def write_json(file_path, payload):
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)

def claim_path_for(job_id):
    return CLAIMS / f'{job_id}.claim.json'

def result_path_for(job_id):
    return OUTPUTS / job_id / 'result.json'

def output_path_for(job_id):
    return OUTPUTS / job_id / 'output.png'

def judge_output_path_for(job_id):
    return OUTPUTS / job_id / 'judge_output.json'

def find_next_job():
    for job_file in sorted(JOB_INBOX.glob('*.json')):
        job_id = job_file.stem
        if claim_path_for(job_id).exists():
            continue
        if result_path_for(job_id).exists():
            continue
        return job_file
    return None

def create_claim(job_id):
    claim_path = claim_path_for(job_id)
    if claim_path.exists():
        return None
    payload = {
        'job_id': job_id,
        'worker_id': 'colab-worker-01',
        'claimed_at': iso_now(),
        'status': 'claimed'
    }
    write_json(claim_path, payload)
    return payload

def unavailable_judge(reason):
    return {
        'source': 'unavailable',
        'status': 'UNAVAILABLE',
        'quality_score': None,
        'overall_score': None,
        'failure_codes': [],
        'notes': [reason],
    }

def parse_gemini_text(text):
    text = (text or '').strip()
    if text.startswith('```json'):
        text = text[7:]
    if text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    return json.loads(text.strip())

def run_live_judge(image_path):
    if not GEMINI_API_KEY:
        return unavailable_judge('GEMINI_API_KEY_MISSING')
    if not image_path.exists():
        return unavailable_judge('OUTPUT_IMAGE_MISSING')
    with open(image_path, 'rb') as handle:
        encoded = base64.b64encode(handle.read()).decode('utf-8')
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}'
    body = {
        'contents': [{
            'parts': [
                {'text': JUDGE_PROMPT},
                {'inline_data': {'mime_type': 'image/png', 'data': encoded}},
            ]
        }],
        'generationConfig': {
            'temperature': 0,
            'response_mime_type': 'application/json'
        }
    }
    request = urllib.request.Request(url, data=json.dumps(body).encode('utf-8'), headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            raw = json.loads(response.read().decode('utf-8'))
        text = raw['candidates'][0]['content']['parts'][0]['text']
        parsed = parse_gemini_text(text)
        parsed['source'] = 'live'
        return parsed
    except Exception as error:
        return unavailable_judge(f'GEMINI_REQUEST_FAILED:{error}')

def render_job(job):
    job_id = job['job_id']
    prompt = job.get('prompt') or job.get('objective') or job.get('goal') or ''
    negative_prompt = job.get('negative_prompt', 'low quality, blurry, distorted, duplicate')
    guidance_scale = float(job.get('guidance_scale', 0.0 if 'turbo' in MODEL_ID else 6.0))
    num_inference_steps = int(job.get('num_inference_steps', 4 if 'turbo' in MODEL_ID else 28))
    seed = int(job.get('seed', 42))
    started_at = iso_now()

    out_dir = OUTPUTS / job_id
    out_dir.mkdir(parents=True, exist_ok=True)
    image_path = output_path_for(job_id)
    result_path = result_path_for(job_id)
    judge_path = judge_output_path_for(job_id)

    try:
        generator = torch.Generator(device=DEVICE)
        generator.manual_seed(seed)
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator
        )
        image = result.images[0]
        image.save(image_path)
        judge_output = run_live_judge(image_path)
        write_json(judge_path, judge_output)

        payload = {
            'job_id': job_id,
            'status': 'completed',
            'output_image_path': f'outputs/{job_id}/output.png',
            'judge_output_path': f'outputs/{job_id}/judge_output.json',
            'judge_output': judge_output,
            'validator_result': {
                'passed': image_path.exists(),
                'signals': [] if image_path.exists() else ['output.png missing after save']
            },
            'artifacts': [
                {
                    'type': 'image',
                    'path': f'outputs/{job_id}/output.png'
                },
                {
                    'type': 'judge_output_json',
                    'path': f'outputs/{job_id}/judge_output.json'
                }
            ],
            'started_at': started_at,
            'finished_at': iso_now(),
            'metadata': {
                'model_id': MODEL_ID,
                'device': DEVICE,
                'seed': seed,
                'num_inference_steps': num_inference_steps,
                'guidance_scale': guidance_scale
            }
        }
        write_json(result_path, payload)
        print('[MIKAGE] completed', job_id, '->', image_path)
        return payload
    except Exception as error:
        judge_output = unavailable_judge('RENDER_FAILED_NO_JUDGE')
        write_json(judge_path, judge_output)
        payload = {
            'job_id': job_id,
            'status': 'failed',
            'judge_output_path': f'outputs/{job_id}/judge_output.json',
            'judge_output': judge_output,
            'error': {
                'code': 'RENDER_FAILED',
                'message': str(error)
            },
            'started_at': started_at,
            'finished_at': iso_now(),
            'metadata': {
                'traceback': traceback.format_exc()[-4000:]
            }
        }
        write_json(result_path, payload)
        print('[MIKAGE] failed', job_id, '->', error)
        return payload

print('[MIKAGE] worker loop started')
while True:
    job_file = find_next_job()
    if job_file is None:
        time.sleep(2)
        continue

    job = safe_read_json(job_file)
    job_id = job.get('job_id') or job_file.stem
    claim = create_claim(job_id)
    if claim is None:
        time.sleep(1)
        continue

    print('[MIKAGE] claimed', job_id, '->', claim_path_for(job_id))
    render_job(job)